# Step 0.1 — Preprocessing: input data (land mask, MIRCA2000 1°, ERA5 1°)

Single source for everything that enters the SVM, built from raw data:

1. **Land mask 5 arc-min**: Natural Earth `land_10`, pixel-centre test.
2. **Land mask 1°**: a 1° cell is land if ≥ 25 % of its 144 five-arc-min pixels are land.
3. **MIRCA2000 1°**: pixels kept only on land in land 1° cells. A pixel is *cropped* if > 5 % of its area is
   harvested. A 1° cell is *cropped* if > 5 % of its kept land pixels are cropped.
   Calendars (sowing / maturity, DOY) are area-weighted circular means over the cropped pixels.
   Wheat is split into winter (sowing Sep–Dec) and spring (sowing Feb–May) at sub-crop level.
4. **ERA5 0.25° → 1°**: ERA5 nodes are kept only if they are land **and** lie in a land 1° cell (same land mask
   as MIRCA). The 1° value is the mean of the kept nodes in the half-open cell `[c-0.5, c+0.5)`, i.e. 4 × 4 nodes.

Output folder: `1_SVM/0_preprocessing_masks/outputs_input_data_1deg/`.


In [1]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr, regionmask
import scipy.sparse as sp
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
CAL_FILE  = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "growing_periods_listed" / "CELL_SPECIFIC_CROPPING_CALENDARS.TXT.gz"
CELL_AREA_FILE = BASE / "0_Input_data" / "masks_crops" / "MIRCA_2000" / "cell_area_grid" / "cell_area_ha.npy"
ERA5_DIR  = cu.data_dir(CFG, "era5")  # data path: see 00_config_and_inventory.ipynb
OUT = BASE / "1_SVM" / "0_preprocessing_masks" / "outputs_input_data_1deg"
OUT.mkdir(parents=True, exist_ok=True)

LAT0, LAT1, LON0, LON1 = 35, 71, -25, 44          # SVM 1° grid (integer centres)
NLAT, NLON = LAT1 - LAT0 + 1, LON1 - LON0 + 1
NCELL = NLAT * NLON
LAND1_THR = 0.25                                  # 1° land cell: >= 25 % land
CROP_THR = 0.05                                   # cropped pixel and cropped cell: > 5 %
lat1 = np.arange(LAT0, LAT1 + 1); lon1 = np.arange(LON0, LON1 + 1)
print("1° grid:", NLAT, "x", NLON, "=", NCELL, "cells")

C:\Users\elisa\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


1° grid: 37 x 70 = 2590 cells


## 1. Land mask at 5 arc-min (Natural Earth) and at 1°

In [2]:
NROW, NCOL = 2160, 4320
ROWS, COLS = np.arange(NROW), np.arange(NCOL)
lat5 = 90 - (ROWS + 0.5) / 12                     # 5-arc-min pixel centres
lon5 = -180 + (COLS + 0.5) / 12
lat5_i = np.floor(lat5 + 0.5).astype(int)         # parent 1° integer centre
lon5_i = np.floor(lon5 + 0.5).astype(int)
R5 = np.where((lat5_i >= LAT0) & (lat5_i <= LAT1))[0]
C5 = np.where((lon5_i >= LON0) & (lon5_i <= LON1))[0]
CELL5 = (lat5_i[R5] - LAT0)[:, None] * NLON + (lon5_i[C5] - LON0)[None, :]
AREA5 = np.load(CELL_AREA_FILE)[np.ix_(R5, C5)]   # ha per 5-arc-min pixel

land_rg = regionmask.defined_regions.natural_earth_v5_0_0.land_10
land5 = np.isfinite(land_rg.mask(lon5[C5], lat5[R5]).values)
land_frac = np.bincount(CELL5.ravel(), weights=land5.ravel().astype(float), minlength=NCELL) / 144.0
LAND1 = land_frac >= LAND1_THR
keep5 = land5 & LAND1[CELL5]                      # kept 5-arc-min land pixels

land1_2d = LAND1.reshape(NLAT, NLON)
xr.Dataset({"land_mask": (("lat", "lon"), land1_2d.astype("int8"))},
           coords={"lat": lat1, "lon": lon1}).to_netcdf(OUT / "land_mask_1deg.nc")
print(f"land 5-arc-min pixels in window: {int(land5.sum())}; land 1° cells: {int(land1_2d.sum())} of {NCELL}")

land 5-arc-min pixels in window: 175844; land 1° cells: 1332 of 2590


## 2. MIRCA2000 1°: crop presence (5 %) and calendars

In [3]:
CAL_CLASSES = {"Wheat": (1, 27), "Maize": (2, 28), "Rice": (3, 29), "Soybean": (8, 34), "Potato": (10, 36)}
# (irrigated code, rainfed code); rainfed = irrigated + 26

cols = ["cell", "row", "col", "lat", "lon", "crop", "subcrop", "area", "start", "end"]
cal = pd.read_csv(CAL_FILE, sep="\t", compression="gzip", usecols=range(10), header=0, names=cols)
allc = [c for v in CAL_CLASSES.values() for c in v]
cal = cal[cal.crop.isin(allc)].copy()
row_map = {r: i for i, r in enumerate(R5)}
col_map = {c: j for j, c in enumerate(C5)}
cal["i"] = (cal.row - 1).map(row_map)
cal["j"] = (cal.col - 1).map(col_map)
cal = cal.dropna(subset=["i", "j"])
cal["i"] = cal.i.astype(int); cal["j"] = cal.j.astype(int)
cal["irr"] = np.where(cal.crop <= 26, "ir", "rf")
name_of = {c: k for k, v in CAL_CLASSES.items() for c in v}
cal["crop_name"] = cal.crop.map(name_of)
print("calendar sub-crop rows in window:", len(cal))


def circ(m):
    return 2 * np.pi * (np.asarray(m, float) - 1) / 12.0      # month -> angle


def month_to_doy(m):
    return (np.asarray(m, float) - 1) * 365.0 / 12.0 + 15.2   # mid-month DOY


presence, sowing, maturity = {}, {}, {}
for crop, g_all in cal.groupby("crop_name"):
    for irr, g in g_all.groupby("irr"):
        if crop == "Wheat":
            seasons = {"winter": g[g.start.isin([9, 10, 11, 12])], "spring": g[g.start.isin([2, 3, 4, 5])]}
        else:
            seasons = {"": g}
        for sname, gs in seasons.items():
            tag = f"{crop}{'-' + sname if sname else ''}_{irr}"
            ii, jj, ar = gs.i.values, gs.j.values, gs.area.values
            A = np.zeros((len(R5), len(C5)))
            np.add.at(A, (ii, jj), ar)                                     # cropped area per pixel (ha)
            # circular accumulators of sub-crop start and end months, area-weighted
            Sx = np.zeros_like(A); Sy = np.zeros_like(A); Ex = np.zeros_like(A); Ey = np.zeros_like(A)
            np.add.at(Sx, (ii, jj), ar * np.cos(circ(gs.start.values)))
            np.add.at(Sy, (ii, jj), ar * np.sin(circ(gs.start.values)))
            np.add.at(Ex, (ii, jj), ar * np.cos(circ(gs.end.values)))
            np.add.at(Ey, (ii, jj), ar * np.sin(circ(gs.end.values)))

            cropped = keep5 & ((A / AREA5) > CROP_THR)                     # pixel-level 5 %
            n_keep = np.bincount(CELL5.ravel(), weights=keep5.ravel().astype(float), minlength=NCELL)
            n_crop = np.bincount(CELL5.ravel(), weights=cropped.ravel().astype(float), minlength=NCELL)
            pres = (n_keep > 0) & (n_crop / np.maximum(n_keep, 1) > CROP_THR)   # cell-level 5 %

            w = np.where(cropped, A, 0.0)

            def cell_month(X, Y):
                sx = np.bincount(CELL5.ravel(), weights=(w * X).ravel(), minlength=NCELL)
                sy = np.bincount(CELL5.ravel(), weights=(w * Y).ravel(), minlength=NCELL)
                return np.mod(np.arctan2(sy, sx), 2 * np.pi) * 12 / (2 * np.pi) + 1.0

            start_m = cell_month(Sx, Sy)
            end_m = cell_month(Ex, Ey)
            sow = np.where(pres, month_to_doy(start_m), np.nan)
            length_days = np.mod(end_m - start_m, 12.0) * 365.0 / 12.0
            mat = np.where(pres, sow + length_days, np.nan)                # maturity may exceed 365

            presence[tag] = pres.reshape(NLAT, NLON).astype("int8")
            sowing[tag] = sow.reshape(NLAT, NLON)
            maturity[tag] = mat.reshape(NLAT, NLON)
            print(f"{tag:22s}: cropped 1° cells {int(pres.sum()):4d}")

for tag in presence:
    xr.Dataset({"presence": (("lat", "lon"), presence[tag]),
                "sowing_doy": (("lat", "lon"), sowing[tag]),
                "maturity_doy": (("lat", "lon"), maturity[tag])},
               coords={"lat": lat1, "lon": lon1}).to_netcdf(OUT / f"mirca2000_1deg_{tag}.nc")
print("saved MIRCA2000 1° products in", OUT)

calendar sub-crop rows in window: 716614
Maize_ir              : cropped 1° cells   82
Maize_rf              : cropped 1° cells  315


Potato_ir             : cropped 1° cells    6
Potato_rf             : cropped 1° cells  101
Rice_ir               : cropped 1° cells   13
Rice_rf               : cropped 1° cells    0
Soybean_ir            : cropped 1° cells    1
Soybean_rf            : cropped 1° cells    3


Wheat-winter_ir       : cropped 1° cells   89
Wheat-spring_ir       : cropped 1° cells    0
Wheat-winter_rf       : cropped 1° cells  718
Wheat-spring_rf       : cropped 1° cells   58


saved MIRCA2000 1° products in C:\Users\elisa\OneDrive - Politecnico di Milano\PHD_codes\crop_suitability_code_bdap\1_SVM\0_preprocessing_masks\outputs_input_data_1deg


## 3. ERA5 0.25° → 1° (land nodes in land 1° cells only)

ERA5 nodes: 149 × 281 at 0.25°. A node is kept if it is land (Natural Earth at the node) **and** its parent
1° cell is land. The 1° value is the arithmetic mean of the kept nodes in `[c-0.5, c+0.5)`.

In [4]:
ERA5_LAT = np.round(np.arange(35.0, 72.0 + 1e-6, 0.25), 2)     # 149 nodes
ERA5_LON = np.round(np.arange(-25.0, 45.0 + 1e-6, 0.25), 2)     # 281 nodes
nlat_e, nlon_e = len(ERA5_LAT), len(ERA5_LON)


def parent_index(v, lo, hi):
    p = np.floor(v + 0.5).astype(int)           # parent integer centre: [c-0.5, c+0.5)
    ok = (p >= lo) & (p <= hi)
    return np.where(ok, p - lo, -1), ok


pi_lat, ok_lat = parent_index(ERA5_LAT, LAT0, LAT1)
pi_lon, ok_lon = parent_index(ERA5_LON, LON0, LON1)

land_e = np.isfinite(land_rg.mask(ERA5_LON, ERA5_LAT).values)   # (lat, lon) node land flags
rows, cols = [], []
for a in range(nlat_e):
    if not ok_lat[a]:
        continue
    for b in range(nlon_e):
        if not ok_lon[b]:
            continue
        cell = pi_lat[a] * NLON + pi_lon[b]
        rows.append(cell); cols.append(a * nlon_e + b)
AGG = sp.csr_matrix((np.ones(len(rows)), (rows, cols)), shape=(NCELL, nlat_e * nlon_e))

parent_land = np.zeros((nlat_e, nlon_e), bool)
for a in range(nlat_e):
    for b in range(nlon_e):
        if ok_lat[a] and ok_lon[b]:
            parent_land[a, b] = LAND1[pi_lat[a] * NLON + pi_lon[b]]
keep_e = land_e & parent_land
KEEP_FLAT = keep_e.ravel().astype(float)
print(f"ERA5 nodes {nlat_e * nlon_e}; land nodes {int(land_e.sum())}; kept {int(keep_e.sum())}")

ERA5 nodes 41869; land nodes 19712; kept 18911


In [5]:
VARS = ["tasmax", "tasmin", "pr", "dswrf"]


def era5_to_1deg(var):
    ds = xr.open_dataset(cu.data_file(CFG, "era5", var))
    time = ds.time.values
    years = pd.DatetimeIndex(time).year.values
    chunks = []
    for yr in np.unique(years):                  # year by year: bounded memory on this machine
        idx = np.where(years == yr)[0]
        X = ds[var].isel(time=idx).values.astype(np.float64).reshape(len(idx), -1)
        valid = np.isfinite(X).astype(float) * KEEP_FLAT
        Xk = np.where(np.isfinite(X), X, 0.0) * KEEP_FLAT
        s = (AGG @ Xk.T).T
        n = (AGG @ valid.T).T
        mean = np.where(n > 0, s / np.maximum(n, 1), np.nan)
        chunks.append(mean.reshape(len(idx), NLAT, NLON).astype(np.float32))
    ds.close()
    data = np.concatenate(chunks, axis=0)
    return xr.DataArray(data, dims=("time", "lat", "lon"),
                        coords={"time": time, "lat": lat1, "lon": lon1}, name=var)


for var in VARS:
    da = era5_to_1deg(var)
    da.to_dataset().to_netcdf(OUT / f"ERA5_1deg_{var}_1981_2015_daily.nc")
    print(var, da.shape, "saved")

tasmax (12783, 37, 70) saved


tasmin (12783, 37, 70) saved


pr (12783, 37, 70) saved


dswrf (12783, 37, 70) saved


## 4. Checks

In [6]:
for var in VARS:
    d = xr.open_dataset(OUT / f"ERA5_1deg_{var}_1981_2015_daily.nc")[var]
    has = d.notnull().any("time").values
    print(f"{var:7s}: shape {d.shape}; cells with data {int(has.sum())}; "
          f"with data outside land mask {int((has & ~land1_2d).sum())}")
    d.close()

tasmax : shape (12783, 37, 70); cells with data 1332; with data outside land mask 0


tasmin : shape (12783, 37, 70); cells with data 1332; with data outside land mask 0


pr     : shape (12783, 37, 70); cells with data 1332; with data outside land mask 0


dswrf  : shape (12783, 37, 70); cells with data 1332; with data outside land mask 0
